# Pipeline d'extraction et de traitement OCR - Domiciliations
Version consolidée à partir des captures d'écran.

In [ ]:
# Installation du runtime minimal compatible Qwen3.6-VL-FP8 (cf. bilans V13c)
# %pip install -q -U 'transformers>=4.57.0' accelerate

import sys
from importlib import metadata

REQUIRED_PACKAGES = {
    "torch": "2.0",
    "transformers": "4.57",
    "accelerate": "0.30",
    "PyMuPDF": "1.23",
    "Pillow": "9.0",
    "openpyxl": "3.1",
    "pandas": "1.5",
    "psutil": "5.9",
}

print("Python :", sys.version.replace("\n", " "))
print("\nPackages détectés :")
missing = []
for package_name, minimum in REQUIRED_PACKAGES.items():
    try:
        version = metadata.version(package_name)
        print(f"  {package_name:15s} {version:12s} | minimum conseillé {minimum}")
    except metadata.PackageNotFoundError:
        missing.append(package_name)
        print(f"  {package_name:15s} ABSENT")

if missing:
    raise RuntimeError(
        "Packages manquants : " + ", ".join(missing) +
        ". Installer uniquement ces packages dans l'environnement Domino."
    )

print("\n✅ Vérification des packages terminée sans modification de l'environnement")

In [ ]:
import gc
import hashlib
import json
import math
import re
import sys
import time
import calendar
from collections import defaultdict
from datetime import date, datetime, timedelta
from pathlib import Path

import fitz
import numpy as np
import pandas as pd
import psutil
import torch
from PIL import Image
from openpyxl import Workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter
from transformers import AutoProcessor, AutoModelForImageTextToText

print("✅ Imports OK")
print("Python       :", sys.version.split()[0])
print("PyMuPDF      :", fitz.__doc__.splitlines()[0] if fitz.__doc__ else "chargé")
print("Torch        :", torch.__version__)
print("CUDA dispo   :", torch.cuda.is_available())
print("GPU          :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Aucun")

In [ ]:
MODEL_PATH = '/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.6-27B-FP8/main'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# -----------------------------------------------------------------------------
# Rendu adaptatif
# -----------------------------------------------------------------------------
# Les contrats et engagements sont dactylographiés : un rendu modéré suffit
# et divise le nombre de tokens image par rapport à la V7.0.
PDF_ZOOM = 3.0
IMAGE_MAX_SIZE = 1400

# La planche « permis de travail » est un scan dégradé, bilingue, avec des
# valeurs manuscrites ou tamponnées : elle est re-rendue à la demande.
PDF_ZOOM_HAUTE_DEF = 4.5
IMAGE_MAX_SIZE_HAUTE_DEF = 2200

MIN_PIXELS = 4 * 32 * 32
# Le plafond du processor doit couvrir le rendu haute définition.
MAX_PIXELS = 2600 * 32 * 32

BLANK_THRESHOLD = 0.995
GPU_BATCH_SIZE_CLASSIFICATION = 2
GPU_BATCH_SIZE_EXTRACTION = 1
MAX_NEW_TOKENS_CLASSIFICATION = 100
MAX_NEW_TOKENS_EXTRACTION = 1700

# -----------------------------------------------------------------------------
# Escalade d'extraction sur les pages difficiles
# -----------------------------------------------------------------------------
# Types de page traités d'emblée en haute définition recadrée.
TYPES_HAUTE_DEFINITION = {"TITRE_TRAVAIL", "PERMIS_TRAVAIL_COUVERTURE"}

# Taux de remplissage au-dessus duquel on arrête d'escalader.
SEUIL_REMPLISSAGE_OK = 0.70

INPUT_DIR = Path('/mnt/data/domiciliations_in/DOM TOSYALI 2026')
OUTPUT_DIR = Path('/mnt/data/domiciliations_out')
JSON_DIR = OUTPUT_DIR / 'json_dossiers'
LOG_PATH = OUTPUT_DIR / 'pipeline_domiciliations.log'
EXCEL_PATH = OUTPUT_DIR / f"domiciliations_{datetime.now().strftime('%Y%m%d_%H%M')}.xlsx"
MASTER_JSON_PATH = OUTPUT_DIR / 'domiciliations_master.json'
PRORATA_MODE = 'CALENDAR_DAYS'
GENERER_MOIS_COMPLETS = True
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
JSON_DIR.mkdir(parents=True, exist_ok=True)
INPUT_DIR.mkdir(parents=True, exist_ok=True)
pdfs = sorted(INPUT_DIR.glob('*.pdf'))
print(f'Device               : {DEVICE}')
print(f'PDFs détectés        : {len(pdfs)}')
print(f'Entrée               : {INPUT_DIR}')
print(f'Sortie               : {OUTPUT_DIR}')
print(f'Prorata retenu       : {PRORATA_MODE}')
CLASSIFICATION_THRESHOLD = 0.80

PIPELINE_VERSION = "DOM_V7_1_QWEN3_6_VL_27B_FP8"
DOM_REFERENCE_EXCEL = Path("/mnt/data/fichier_domiciliations.xlsx")
PREDOM_REFERENCE_EXCEL = Path("/mnt/data/fichier_predomiciliations.xlsx")
NAME_MATCH_THRESHOLD = 0.86
DATE_TOLERANCE_DAYS = 5
EXCEL_AMOUNT_FORMAT = "0.00"
AMOUNT_FIELDS = {
    "DOM_SALAIRE_NET_MENSUEL", "DOM_PART_TRANSFERABLE",
    "DOM_MONTANT_TOTAL_DOMICILIE", "CTR_SALAIRE_BRUT",
    "CTR_SALAIRE_NET", "CTS_SALAIRE_NET",
    "CTS_SALAIRE_NET_ANCIEN",
    "CTS_PART_TRANSFERABLE", "CTS_PART_PAYABLE_DZD",
}

In [ ]:
if DEVICE != "cuda":
    raise RuntimeError("Ce pipeline nécessite un GPU CUDA.")

torch.backends.cuda.matmul.allow_tf32 = True

print("Chargement du processor...")
t0 = time.time()
processor = AutoProcessor.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
    min_pixels=MIN_PIXELS,
    max_pixels=MAX_PIXELS,
)
processor.tokenizer.padding_side = "left"

print("Chargement du modèle FP8...")
try:
    from transformers.integrations.finegrained_fp8 import FineGrainedFP8Config as FP8Config
except ImportError:
    from transformers import FineGrainedFP8Config as FP8Config

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_PATH,
    dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True,
    quantization_config=FP8Config(dequantize=True),
)
model.eval()

print(f"✅ Modèle chargé en {time.time() - t0:.1f}s | dtype=bfloat16 (FP8 déquantifié)")
print(f"VRAM allouée : {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def resize_image(img, max_side=IMAGE_MAX_SIZE):
    w, h = img.size
    if max(w, h) <= max_side:
        return img
    ratio = max_side / max(w, h)
    return img.resize((int(w * ratio), int(h * ratio)), Image.LANCZOS)

def white_ratio(image):
    arr = np.array(image.convert("L"))
    return float((arr > 245).sum() / arr.size)

def is_blank(image, threshold=BLANK_THRESHOLD):
    return white_ratio(image) >= threshold

def pdf_to_pages(path, zoom=PDF_ZOOM):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"PDF introuvable : {path}")
    if path.stat().st_size == 0:
        raise ValueError(f"PDF vide : {path}")

    pages = []
    doc = fitz.open(str(path))
    try:
        page_count = int(doc.page_count)
        if page_count <= 0:
            raise ValueError(f"PyMuPDF ne détecte aucune page dans : {path.name}")

        matrix = fitz.Matrix(zoom, zoom)
        for i in range(page_count):
            page = doc.load_page(i)
            pix = page.get_pixmap(matrix=matrix, alpha=False)

            if pix.width <= 0 or pix.height <= 0 or not pix.samples:
                raise ValueError(
                    f"Rendu image vide : {path.name}, page {i + 1}"
                )

            img = Image.frombytes(
                "RGB",
                (pix.width, pix.height),
                pix.samples
            )
            img = resize_image(img)

            pages.append({
                "index": i,
                "page_num": i + 1,
                "image": img,
                "width": img.width,
                "height": img.height,
                "white_ratio": round(white_ratio(img), 6),
            })
    finally:
        doc.close()

    if len(pages) != page_count:
        raise RuntimeError(
            f"Conversion incomplète de {path.name}: "
            f"{len(pages)} image(s) pour {page_count} page(s)"
        )
    return pages

In [ ]:
def parse_json_response(text):
    if not text:
        return {}

    clean = str(text).strip()
    clean = re.sub(r"^```(?:json)?", "", clean, flags=re.I).strip()
    clean = re.sub(r"```$", "", clean).strip()

    match = re.search(r"\{.*\}", clean, flags=re.S)
    if not match:
        return {}

    candidate = match.group(0)
    attempts = [
        candidate,
        re.sub(r",\s*([}\]\]])", r"\1", candidate),
    ]

    for attempt in attempts:
        try:
            parsed = json.loads(attempt)
            return parsed if isinstance(parsed, dict) else {}
        except Exception:
            continue
    return {}

def log(message):
    line = f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - {message}"
    print(line)
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(line + "\n")

print("✅ Utilitaires PDF/JSON OK")

In [ ]:
# -----------------------------------------------------------------------------
# Rendu haute définition et recadrage (planche « permis de travail »)
# -----------------------------------------------------------------------------

def crop_region(image, haut=0.0, bas=1.0, gauche=0.0, droite=1.0):
    """
    Recadre une image par fractions de sa hauteur et de sa largeur.
    Les fractions sont exprimées entre 0.0 et 1.0.
    """
    largeur, hauteur = image.size

    x0 = int(max(0.0, min(1.0, gauche)) * largeur)
    x1 = int(max(0.0, min(1.0, droite)) * largeur)
    y0 = int(max(0.0, min(1.0, haut)) * hauteur)
    y1 = int(max(0.0, min(1.0, bas)) * hauteur)

    if x1 <= x0 or y1 <= y0:
        return image

    return image.crop((x0, y0, x1, y1))

def render_page_region(
    pdf_path,
    page_index,
    zoom=PDF_ZOOM_HAUTE_DEF,
    max_side=IMAGE_MAX_SIZE_HAUTE_DEF,
    crop=None,
):
    """
    Re-rend une page du PDF en haute définition, éventuellement recadrée.
    Le recadrage est appliqué AVANT le redimensionnement.
    """
    pdf_path = Path(pdf_path)
    doc = fitz.open(str(pdf_path))
    try:
        page = doc.load_page(int(page_index))
        pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom), alpha=False)

        if pix.width <= 0 or pix.height <= 0 or not pix.samples:
            raise ValueError(
                f"Rendu haute définition vide : {pdf_path.name}, "
                f"page {int(page_index) + 1}"
            )

        img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
    finally:
        doc.close()

    if crop:
        img = crop_region(img, *crop)

    return resize_image(img, max_side=max_side)

def taux_remplissage(data, champs_attendus):
    """
    Part des champs attendus effectivement renseignés.
    """
    if not champs_attendus:
        return 1.0

    remplis = sum(
        1
        for champ in champs_attendus
        if (data or {}).get(champ) not in (None, "")
    )
    return round(remplis / len(champs_attendus), 4)

print("✅ Rendu haute définition et recadrage OK")

In [ ]:
FRONTIERE_ZONE_RECHERCHE = (0.28, 0.66)  # plage explorée
FRONTIERE_SEUIL_ENCRE = 0.004             # densité en dessous de laquelle une ligne est considérée vide
FRONTIERE_DEFAULT = 0.47                  # repli si aucune bande nette

def detecter_frontiere_documents(
    image,
    zone=FRONTIERE_ZONE_RECHERCHE,
    seuil_encre=FRONTIERE_SEUIL_ENCRE,
    defaut=FRONTIERE_DEFAULT,
):
    """
    Renvoie la fraction de hauteur séparant les deux documents d'une planche.
    """
    try:
        arr = np.array(image.convert("L"))
    except Exception:
        return defaut

    hauteur = arr.shape[0]
    if hauteur < 10:
        return defaut

    densite_encre = (arr < 200).sum(axis=1) / max(arr.shape[1], 1)

    y_min = int(max(0.0, zone[0]) * hauteur)
    y_max = int(min(1.0, zone[1]) * hauteur)
    if y_max <= y_min:
        return defaut

    bandes = []
    debut = None
    for y in range(y_min, y_max):
        vide = densite_encre[y] < seuil_encre
        if vide and debut is None:
            debut = y
        elif not vide and debut is not None:
            bandes.append((debut, y))
            debut = None
    if debut is not None:
        bandes.append((debut, y_max))

    if not bandes:
        return defaut

    debut, fin = max(bandes, key=lambda b: b[1] - b[0])

    # Une bande trop fine n'est qu'un interligne, pas une coupure.
    if (fin - debut) / hauteur < 0.015:
        return defaut

    return round((debut + fin) / 2 / hauteur, 4)

def crops_planche_permis(image, marge=0.02):
    """
    Construit les recadrages de la planche à partir de la frontière détectée.
    """
    frontiere = detecter_frontiere_documents(image)

    bas_titre = min(1.0, frontiere + marge)
    haut_couverture = max(0.0, frontiere - marge)

    return {
        "frontiere": frontiere,
        "titre": (0.00, bas_titre, 0.00, 1.00),
        "colonne_identite": (0.00, bas_titre, 0.44, 1.00),
        "colonne_poste": (0.00, bas_titre, 0.00, 0.56),
        "couverture": (haut_couverture, 1.00, 0.00, 1.00),
    }

print("✅ Détection automatique de la frontière de planche OK")

In [ ]:
def canonical_checkpoint_path(pdf_path):
    """
    Un seul fichier JSON par PDF : <nom_du_pdf_sans_extension>.json
    """
    return JSON_DIR / f"{pdf_path.stem}.json"

def checkpoint_is_complete(dossier, pdf_path):
    """
    Un checkpoint est réutilisable seulement s'il correspond au PDF
    et contient une extraction complète avec au moins une page.
    """
    if not isinstance(dossier, dict):
        return False

    stats = dossier.get("stats") or {}
    page_records = dossier.get("page_records") or []

    if dossier.get("source_file") != pdf_path.name:
        return False
    if int(stats.get("pages", 0) or 0) <= 0:
        return False
    if not page_records:
        return False

    stored_hash = dossier.get("source_sha256")
    if not stored_hash:
        return False

    try:
        return stored_hash == sha256_file(pdf_path)
    except Exception:
        return False

def load_existing_checkpoint(pdf_path):
    """
    Cherche d'abord le JSON canonique, puis les anciens JSON suffixés
    par l'empreinte.
    """
    canonical = canonical_checkpoint_path(pdf_path)
    candidates = [canonical] + sorted(
        JSON_DIR.glob(f"{pdf_path.stem}_*.json"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )

    seen = set()
    for candidate in candidates:
        if candidate in seen or not candidate.exists():
            continue
        seen.add(candidate)

        try:
            dossier = json.loads(candidate.read_text(encoding="utf-8"))
        except Exception:
            continue

        if not checkpoint_is_complete(dossier, pdf_path):
            continue

        if candidate != canonical:
            canonical.write_text(
                json.dumps(dossier, ensure_ascii=False, indent=2, default=str),
                encoding="utf-8",
            )

        for duplicate in JSON_DIR.glob(f"{pdf_path.stem}_*.json"):
            if duplicate.exists():
                duplicate.unlink()

        return dossier

    return None

def enrich_dossier_row_with_stats(dossier, statut_traitement):
    """
    Ajoute au niveau dossier les indicateurs visibles dans Excel.
    """
    row = dossier.get("dossier_row") or {}
    stats = dossier.get("stats") or {}

    row.update({
        "STATUT_TRAITEMENT_PIPELINE": statut_traitement,
        "TEMPS_ECOULE_DOSSIER_S": round(float(stats.get("elapsed_s", 0) or 0), 2),
        "TOKENS_IN_DOSSIER": int(stats.get("tokens_in", 0) or 0),
        "TOKENS_OUT_DOSSIER": int(stats.get("tokens_out", 0) or 0),
        "TOKENS_TOTAL_DOSSIER": int(stats.get("tokens_total", 0) or 0),
    })

    dossier["dossier_row"] = row
    return dossier

def format_duration(seconds):
    seconds = max(0, int(round(float(seconds or 0))))
    hours, rem = divmod(seconds, 3600)
    minutes, secs = divmod(rem, 60)
    if hours:
        return f"{hours:02d}:{minutes:02d}:{secs:02d}"
    return f"{minutes:02d}:{secs:02d}"

def print_pipeline_header(total_pdfs):
    print(
        f"\n{PIPELINE_VERSION} | {total_pdfs} PDF détecté(s)\n",
        flush=True,
    )

def print_compact_progress(
    position, total, pdf_name, pages, skipped, tokens_in, tokens_out, elapsed_s, pipeline_start,
):
    elapsed_global = time.time() - pipeline_start
    avg = elapsed_global / max(position, 1)
    eta = avg * max(total - position, 0)
    status = "SKIP" if skipped else "TRAITE"

    print(
        f"[{position}/{total}] "
        f"{pdf_name} | "
        f"pages={int(pages or 0)} | "
        f"{status} | "
        f"IN={int(tokens_in or 0)} | "
        f"OUT={int(tokens_out or 0)} | "
        f"{float(elapsed_s or 0):.2f}s | "
        f"ETA={format_duration(eta)}",
        flush=True,
    )

In [ ]:
def apply_template(messages):
    """Qwen3 : désactive le mode 'thinking' si supporté."""
    try:
        return processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        return processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

def ask_single(prompt, image, max_new_tokens):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    text_in = apply_template(messages)
    inputs = processor(
        text=text_in,
        images=image,
        return_tensors="pt",
    ).to(DEVICE)

    t0 = time.time()
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.0,
            pad_token_id=processor.tokenizer.eos_token_id,
        )
    torch.cuda.synchronize()

    generated = out[0][inputs["input_ids"].shape[1]:]
    text = processor.decode(
        generated,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    )

    return {
        "text": text,
        "tokens_in": int(inputs["input_ids"].shape[1]),
        "tokens_out": int(len(generated)),
        "elapsed_s": round(time.time() - t0, 3),
    }

def ask_batch(prompt, images, max_new_tokens):
    if not images:
        return []
    if len(images) == 1:
        return [ask_single(prompt, images[0], max_new_tokens)]

    texts_in = []
    for image in images:
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": prompt},
                ],
            }
        ]
        texts_in.append(apply_template(messages))

    inputs = processor(
        text=texts_in,
        images=images,
        return_tensors="pt",
        padding=True,
    ).to(DEVICE)

    t0 = time.time()
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.0,
            pad_token_id=processor.tokenizer.eos_token_id,
        )
    torch.cuda.synchronize()

    if out.shape[0] != len(images):
        raise RuntimeError(
            f"Réponses VLM incohérentes : {out.shape[0]} sortie(s) "
            f"pour {len(images)} image(s)"
        )

    elapsed = time.time() - t0
    input_width = inputs["input_ids"].shape[1]
    attention_mask = inputs.get("attention_mask")
    results = []

    for i in range(len(images)):
        generated = out[i][input_width:]
        text = processor.decode(
            generated,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )
        tokens_in = (
            int(attention_mask[i].sum().item())
            if attention_mask is not None
            else int(input_width)
        )
        results.append({
            "text": text,
            "tokens_in": tokens_in,
            "tokens_out": int(len(generated)),
            "elapsed_s": round(elapsed / len(images), 3),
        })

    return results

print("✅ Inférence single/batch OK")

In [ ]:
PROMPT_CLASSIFICATION = """
Analyse le titre, les en-têtes, la mise en page et les blocs visuels
de cette page.

Classe la page dans exactement une seule catégorie :

- ENGAGEMENT_DOMICILIATION
- CONTRAT_TRAVAIL
- CONTRAT_SPECIFIQUE
- TITRE_TRAVAIL
- PERMIS_TRAVAIL_COUVERTURE
- AUTRE

REGLE DE PRIORITE ABSOLUE :
Certaines pages contiennent DEUX documents superposés : un titre de
travail bilingue dans la moitié haute et une couverture de permis de
travail dans la moitié basse.
Dans ce cas, classer TOUJOURS la page en TITRE_TRAVAIL.
Le grand titre « رخصة عمل / Permis de Travail » de la moitié basse ne
doit JAMAIS l'emporter sur un bloc d'identité présent dans la moitié
haute.

Règles de classification :

- ENGAGEMENT_DOMICILIATION :
  le titre contient « ENGAGEMENT DE DOMICILIATION »
  ou « CONTRAT DES SALARIES ETRANGERS ».

- CONTRAT_TRAVAIL :
  le titre contient « CONTRAT DE TRAVAIL A DUREE DETERMINEE ».

- CONTRAT_SPECIFIQUE :
  le titre contient « CONTRAT DE TRAVAIL SPECIFIQUE
  A LA MAIN D'OEUVRE ETRANGERE ».

- TITRE_TRAVAIL :
  la page contient un bloc d'identité du travailleur, reconnaissable à
  AU MOINS DEUX des éléments suivants :
  - une photographie d'identité ;
  - les libellés « Nom » et « Prénom » suivis de valeurs ;
  - les libellés « Date de naissance » / « Lieu de naissance » ;
  - le libellé « Date d'entrée en Algérie » ;
  - des libellés arabes d'identité (اللقب، الاسم، تاريخ الميلاد).
  Cette catégorie s'applique même si la page comporte aussi des cachets,
  un QR code, du texte de loi ou un second document en dessous.

- PERMIS_TRAVAIL_COUVERTURE :
  UNIQUEMENT si la page ne contient AUCUN bloc d'identité du travailleur
  et se limite au titre « Permis de Travail », au numéro de série et aux
  extraits de loi.

- AUTRE :
  aucun type ne correspond clairement.

Ne te base jamais uniquement sur le numéro de page.

Retourne uniquement ce JSON :
{
  "type_document": "TYPE",
  "confidence": 0.00,
  "titre_detecte": "TITRE BRUT OU null",
  "bloc_identite_present": true
}
"""

COMMON_RAW_RULES = """
Tu analyses une seule image, qui peut être une page entière ou un
recadrage d'une page.

REGLES OBLIGATOIRES :
1. Extraire uniquement les champs demandés.
2. Pour chaque champ, rechercher le libellé indiqué.
3. Recopier uniquement la valeur située juste après le libellé :
   - sur la même ligne ;
   - ou immédiatement sur la ligne suivante si la valeur continue.
4. Conserver la valeur exactement comme elle apparaît :
   espaces, ponctuation, séparateurs, format de date et format de montant.
5. Ne corrige pas l'orthographe.
6. Ne normalise pas les dates.
7. Ne normalise pas les montants.
8. Ne sépare pas automatiquement le nom et le prénom.
9. Ne complète pas une valeur partiellement lisible.
10. N'utilise aucune valeur provenant d'une autre page.
11. Si le libellé est absent ou la valeur illisible, retourne null.
12. N'invente jamais une valeur.
13. Retourne uniquement un objet JSON valide, sans commentaire.
14. Un champ absent du recadrage que tu analyses doit valoir null.
    Ne devine pas ce qui se trouve hors de l'image.
"""

PROMPT_ENGAGEMENT = COMMON_RAW_RULES + """
TYPE ATTENDU : ENGAGEMENT_DOMICILIATION

Extrais exactement les clés suivantes :
{
  "DOM_NOM_RAISON_SOCIAL_CLIENT": null,
  "DOM_COMPTE_LOCAL": null,
  "DOM_ADRESSE_CLIENT": null,
  "DOM_AGENCE_DOMICILIATAIRE": null,
  "DOM_NUMERO_CONTRAT": null,
  "DOM_DUREE_CONTRAT_MOIS": null,
  "DOM_DATE_DEBUT_CONTRAT": null,
  "DOM_DATE_FIN_CONTRAT": null,
  "DOM_NOM_RAISON_SOCIAL_EMPLOYEUR": null,
  "DOM_ADRESSE_EMPLOYEUR": null,
  "DOM_SALAIRE_NET_MENSUEL": null,
  "DOM_PART_TRANSFERABLE": null,
  "DOM_TAUX_TRANSFERABLE": null,
  "DOM_MONTANT_TOTAL_DOMICILIE": null,
  "DOM_DATE_SIGNATURE": null
}

Libellés et règles :
- DOM_NOM_RAISON_SOCIAL_CLIENT :
  valeur après « Nom et raison sociale ».
  La valeur peut être répartie en deux colonnes (nom puis prénom) :
  recopier les deux, séparés par un espace.
- DOM_COMPTE_LOCAL :
  valeur après « N de compte » ou « N° de compte ».
- DOM_ADRESSE_CLIENT :
  valeur après la première occurrence de « Adresse »
  dans la section « Identification du client ».
- DOM_AGENCE_DOMICILIATAIRE :
  valeur après « Agence domiciliataire », en haut à droite.
- DOM_NUMERO_CONTRAT :
  valeur après « Numéro du contrat ».
- DOM_DUREE_CONTRAT_MOIS :
  valeur après « Durée du contrat » ou « Duré du contrat ».
- DOM_DATE_DEBUT_CONTRAT :
  valeur après « Date de début de contrat ».
- DOM_DATE_FIN_CONTRAT :
  valeur après « Date de fin de contrat ».
- DOM_NOM_RAISON_SOCIAL_EMPLOYEUR :
  valeur après « Nom et raison sociale de l'Employeur ».
- DOM_ADRESSE_EMPLOYEUR :
  valeur après « Adresse de l'Employeur ».
  Continuer sur la ligne suivante si l'adresse se poursuit.
- DOM_SALAIRE_NET_MENSUEL :
  valeur après « Salaire net mensuel ».
- DOM_PART_TRANSFERABLE :
  valeur après « Montant de la part transférable ».
- DOM_TAUX_TRANSFERABLE :
  valeur après « Pourcentage en regard du salaire net mensuel ».
- DOM_MONTANT_TOTAL_DOMICILIE :
  valeur après « Montant domicilié en DZD ».
  Ce champ est souvent laissé vide : retourner null dans ce cas.
- DOM_DATE_SIGNATURE :
  date manuscrite ou imprimée située près de la mention
  « lu et approuvé », en bas de page.
"""

PROMPT_CONTRAT = COMMON_RAW_RULES + """
TYPE ATTENDU : CONTRAT_TRAVAIL

Extrais exactement les clés suivantes :
{
  "CTR_REFERENCE_DOCUMENT": null,
  "CTR_TYPE": null,
  "CTR_EMPLOYEUR": null,
  "CTR_ACTIVITE_EMPLOYEUR": null,
  "CTR_DUREE_MOIS": null,
  "CTR_DATE_DEBUT_CONTRAT": null,
  "CTR_POSTE": null,
  "CTR_NOM_PRENOM_TRAVAILLEUR": null,
  "CTR_PERE_NOM_PRENOM": null,
  "CTR_MERE_NOM_PRENOM": null,
  "CTR_NATIONALITE": null,
  "CTR_DATE_NAISSANCE": null,
  "CTR_LIEU_PAYS_NAISSANCE": null,
  "CTR_ADRESSE_ALGERIE": null,
  "CTR_QUALIFICATION": null,
  "CTR_NUMERO_PERMIS_TRAVAIL": null,
  "CTR_DATE_DELIVRANCE_PERMIS": null,
  "CTR_DATE_DEBUT_VALIDITE_PERMIS": null,
  "CTR_DATE_FIN_VALIDITE_PERMIS": null,
  "CTR_SALAIRE_BRUT": null,
  "CTR_SALAIRE_NET": null,
  "CTR_AFFILIATION_SS": null,
  "CTR_NUMERO_EMPLOYEUR": null,
  "CTR_DATE_SIGNATURE": null,
  "CTR_REFERENCE_DOMICILIATION": null,
  "CTR_SIGNATURE_TRAVAILLEUR_PRESENTE": null,
  "CTR_SIGNATURE_EMPLOYEUR_PRESENTE": null,
  "CTR_CACHET_EMPLOYEUR_PRESENT": null
}

Libellés et règles :
- CTR_REFERENCE_DOCUMENT :
  référence imprimée dans le coin supérieur gauche,
  par exemple « TR-6166 ».
- CTR_TYPE :
  titre complet du document.
- CTR_EMPLOYEUR :
  valeur après
  « au nom de l'employeur ci-après désigné ».
- CTR_ACTIVITE_EMPLOYEUR :
  valeur après « Nature de l'activité ».
- CTR_DUREE_MOIS :
  valeur après « pour une durée de ».
  et avant « à compter du ».
- CTR_DATE_DEBUT_CONTRAT :
  valeur après « à compter du ».
- CTR_POSTE :
  valeur après « En qualité de ».
- CTR_NOM_PRENOM_TRAVAILLEUR :
  valeur après « A (Mr/Mme) ».
- CTR_PERE_NOM_PRENOM :
  valeur après « Fils de »
  et avant « et de ».
- CTR_MERE_NOM_PRENOM :
  valeur après « et de ».
- CTR_NATIONALITE :
  valeur après « Nationalité ».
- CTR_DATE_NAISSANCE :
  valeur après « Né(e) le »
  et avant « à ».
- CTR_LIEU_PAYS_NAISSANCE :
  valeur après « à » sur la ligne de naissance.
- CTR_ADRESSE_ALGERIE :
  valeur après « Adresse en Algérie ».
- CTR_QUALIFICATION :
  valeur après « Qualification professionnelle ».
- CTR_NUMERO_PERMIS_TRAVAIL :
  valeur après « permis de travail N° ».
  Recopier la référence complète, y compris la partie après « / ».
- CTR_DATE_DELIVRANCE_PERMIS :
  valeur après « Délivré le ».
- CTR_DATE_DEBUT_VALIDITE_PERMIS :
  première date après « Valable du ».
- CTR_DATE_FIN_VALIDITE_PERMIS :
  date après « au » sur la même ligne.
- CTR_SALAIRE_BRUT :
  valeur après « Montant du salaire mensuel brut ».
- CTR_SALAIRE_NET :
  valeur après « Montant du salaire mensuel net ».
- CTR_AFFILIATION_SS :
  valeur après « Affiliation à la sécurité sociale ».
- CTR_NUMERO_EMPLOYEUR :
  valeur après « Employeur ».
- CTR_DATE_SIGNATURE :
  date après « Fait à : Bethioua, le ».
- CTR_REFERENCE_DOMICILIATION :
  dans le cachet « DOMICILIATION IMPORT »,
  recopier les cinq cases dans l'ordre
  et les séparer par « | ».
  Exemple : 2719081|2026.1|48|00119|DZD
  null si ce cachet est absent de la page.
- Contrôles visuels :
- CTR_SIGNATURE_TRAVAILLEUR_PRESENTE :
  true si un tracé manuscrit est visible directement sous
  « Signature du Travailleur Etranger », sinon false.
- CTR_SIGNATURE_EMPLOYEUR_PRESENTE :
  true si un tracé manuscrit est visible directement sous
  « Signature de l'Employeur », sinon false.
- CTR_CACHET_EMPLOYEUR_PRESENT :
  true si une empreinte de cachet est visible dans la zone
  « Signature de l'Employeur », sinon false.
"""

PROMPT_CONTRAT_SPECIFIQUE = COMMON_RAW_RULES + """
TYPE ATTENDU : CONTRAT_SPECIFIQUE

Extrais exactement les clés suivantes :
{
  "CTS_REFERENCE_DOCUMENT": null,
  "CTS_SAP_ID": null,
  "CTS_EMPLOYEUR": null,
  "CTS_ACTIVITE_EMPLOYEUR": null,
  "CTS_DUREE_MOIS": null,
  "CTS_DATE_DEBUT_CONTRAT": null,
  "CTS_POSTE": null,
  "CTS_NOM_PRENOM_TRAVAILLEUR": null,
  "CTS_PERE_NOM_PRENOM": null,
  "CTS_MERE_NOM_PRENOM": null,
  "CTS_NATIONALITE": null,
  "CTS_DATE_NAISSANCE": null,
  "CTS_LIEU_PAYS_NAISSANCE": null,
  "CTS_ADRESSE_ALGERIE": null,
  "CTS_QUALIFICATION": null,
  "CTS_NUMERO_PERMIS_TRAVAIL": null,
  "CTS_DATE_DELIVRANCE_PERMIS": null,
  "CTS_DATE_DEBUT_VALIDITE_PERMIS": null,
  "CTS_DATE_FIN_VALIDITE_PERMIS": null,
  "CTS_LIGNE_SALAIRE_BRUTE": null,
  "CTS_SALAIRE_NET": null,
  "CTS_SALAIRE_NET_ANCIEN": null,
  "CTS_MENTION_AU_LIEU_DE_PRESENTE": null,
  "CTS_PART_TRANSFERABLE": null,
  "CTS_PART_PAYABLE_DZD": null,
  "CTS_NUMERO_SS_PAYS_ORIGINE": null,
  "CTS_NUMERO_SS_ALGERIE": null,
  "CTS_DATE_DOCUMENT": null,
  "CTS_SIGNATURE_TRAVAILLEUR_PRESENTE": null,
  "CTS_SIGNATURE_EMPLOYEUR_PRESENTE": null,
  "CTS_CACHET_EMPLOYEUR_PRESENT": null,
  "CTS_VISA_INSPECTION_TRAVAIL_PRESENT": null
}

Libellés et règles :
- CTS_REFERENCE_DOCUMENT :
  référence imprimée dans le coin supérieur gauche, par exemple « TR-6166 ».
- CTS_SAP_ID :
  valeur après « SAP id - » en haut de page.
- CTS_EMPLOYEUR :
  valeur après « au nom de l'employeur ci-après désigné ».
- CTS_ACTIVITE_EMPLOYEUR :
  valeur après « Nature de l'activité ».
- CTS_DUREE_MOIS :
  valeur après « pour une durée de ».
- CTS_DATE_DEBUT_CONTRAT :
  valeur après « A compter du ».
- CTS_POSTE :
  valeur après « en qualité de ».
- CTS_NOM_PRENOM_TRAVAILLEUR :
  valeur après « A (Mr/Mme) ».
- CTS_PERE_NOM_PRENOM :
  valeur après « fils de »
  et avant « et de ».
- CTS_MERE_NOM_PRENOM :
  valeur après « et de ».
- CTS_NATIONALITE :
  valeur après « Nationalité ».
- CTS_DATE_NAISSANCE :
  valeur après « Né(e) le »
  et avant « à ».
- CTS_LIEU_PAYS_NAISSANCE :
  valeur après « à » sur la ligne de naissance.
- CTS_ADRESSE_ALGERIE :
  valeur après « Adresse en Algérie ».
- CTS_QUALIFICATION :
  valeur après « Qualification professionnelle ».
- CTS_NUMERO_PERMIS_TRAVAIL :
  valeur après « permis de travail N° ».
  Recopier la référence complète, y compris la partie après « / ».
- CTS_DATE_DELIVRANCE_PERMIS :
  valeur après « Délivré le ».
- CTS_DATE_DEBUT_VALIDITE_PERMIS :
  première date après « Valable du ».
- CTS_DATE_FIN_VALIDITE_PERMIS :
  date après « au » sur la même ligne.

--- LIGNE DE SALAIRE : RÈGLE PARTICULIÈRE ---
Cette ligne peut prendre DEUX formes :

Forme A (salaire inchangé) :
« Salaire mensuel de base net : 506,471.38 »

Forme B (augmentation de salaire) :
« Salaire mensuel de base net : 506,471.38 au lieu de 479,274.29 »

- CTS_LIGNE_SALAIRE_BRUTE :
  recopier la ligne ENTIÈRE telle qu'elle apparaît, du libellé
  « Salaire mensuel de base net » jusqu'à la fin de la ligne,
  sans rien retirer.

- CTS_SALAIRE_NET :
  le PREMIER montant de cette ligne, c'est-à-dire celui situé
  immédiatement après « Salaire mensuel de base net : ».
  En forme B, c'est le NOUVEAU salaire, celui placé AVANT
  « au lieu de ». Ne jamais recopier « au lieu de » ni ce qui suit.

- CTS_SALAIRE_NET_ANCIEN :
  le SECOND montant de cette ligne, celui placé APRÈS
  « au lieu de ». C'est l'ANCIEN salaire.
  null si la mention « au lieu de » est absente de cette ligne.

- CTS_MENTION_AU_LIEU_DE_PRESENTE :
  true si la ligne de salaire contient « au lieu de », sinon false.

--- SUITE ---

- CTS_PART_TRANSFERABLE :
  valeur après « la part transférable ».
- CTS_PART_PAYABLE_DZD :
  valeur après « la part payable en dinars algérien ».
- CTS_NUMERO_SS_PAYS_ORIGINE :
  valeur après « Dans le pays d'origine ».
- CTS_NUMERO_SS_ALGERIE :
  valeur après « En Algérie ».
- CTS_DATE_DOCUMENT :
  date après « Fait à : Bethioua, le ».

Contrôles visuels :
- CTS_SIGNATURE_TRAVAILLEUR_PRESENTE :
  true si un tracé manuscrit est visible sous
  « Signature du Travailleur Etranger ».
- CTS_SIGNATURE_EMPLOYEUR_PRESENTE :
  true si un tracé manuscrit est visible sous
  « Signature de l'Employeur ».
- CTS_CACHET_EMPLOYEUR_PRESENT :
  true si une empreinte de cachet est visible dans la zone employeur.
- CTS_VISA_INSPECTION_TRAVAIL_PRESENT :
  true si le bas de page comporte un cachet ou une mention manuscrite
  près de « le présent contrat a été visé par nous ».
"""

PROMPT_TITRE_TRAVAIL = COMMON_RAW_RULES + """
TYPE ATTENDU : TITRE_TRAVAIL
"""

In [ ]:
def extract_one_record(record, pdf_path):
    """
    Extraction d'une page avec escalade progressive.
    """
    doc_type = record["doc_type"]
    prompt = PROMPTS_EXTRACTION[doc_type]
    champs = CHAMPS_ATTENDUS.get(doc_type) or []
    strategies = STRATEGIES_EXTRACTION.get(doc_type) or [{"nom": "STANDARD"}]

    fusion = {}
    strategies_utilisees = []
    textes_bruts = []
    tokens_in = 0
    tokens_out = 0
    elapsed = 0.0

    crops_planche = None
    if doc_type in TYPES_HAUTE_DEFINITION:
        crops_planche = crops_planche_permis(record["image"])
        record["frontiere_planche"] = crops_planche["frontiere"]

    for rang, strategie in enumerate(strategies):
        if rang > 0 and taux_remplissage(fusion, champs) >= SEUIL_REMPLISSAGE_OK:
            break

        crop = strategie.get("crop")
        cle_dynamique = strategie.get("crop_dynamique")
        if cle_dynamique and crops_planche:
            crop = crops_planche[cle_dynamique]

        if crop is None:
            image = record["image"]
        else:
            image = render_page_region(
                pdf_path,
                record["page_num"] - 1,
                zoom=strategie.get("zoom", PDF_ZOOM_HAUTE_DEF),
                max_side=strategie.get("max_side", IMAGE_MAX_SIZE_HAUTE_DEF),
                crop=crop,
            )

        output = ask_single(prompt, image, MAX_NEW_TOKENS_EXTRACTION)
        parsed = parse_json_response(output["text"])

        nouveaux = 0
        for cle, valeur in parsed.items():
            if fusion.get(cle) in (None, "") and valeur not in (None, ""):
                fusion[cle] = valeur
                nouveaux += 1

        tokens_in += output["tokens_in"]
        tokens_out += output["tokens_out"]
        elapsed += output["elapsed_s"]
        textes_bruts.append(f"[{strategie['nom']}] {output['text']}")
        strategies_utilisees.append({
            "nom": strategie["nom"],
            "crop": crop,
            "champs_ajoutes": nouveaux,
            "taux_apres": taux_remplissage(fusion, champs),
        })

    for cle in champs:
        fusion.setdefault(cle, None)

    record["raw_data"] = fusion
    record["extraction_taux_remplissage"] = taux_remplissage(fusion, champs)
    record["extraction_strategies"] = strategies_utilisees
    record["extraction_raw_text"] = "\n".join(textes_bruts)
    record["extraction_tokens_in"] = tokens_in
    record["extraction_tokens_out"] = tokens_out
    record["extraction_elapsed_s"] = round(elapsed, 3)
    record["extraction_status"] = (
        "OK"
        if record["extraction_taux_remplissage"] >= SEUIL_REMPLISSAGE_OK
        else ("PARTIELLE" if fusion else "JSON_VIDE")
    )

    return record

def extract_classified_pages(records, pdf_path):
    for record in records:
        if record["doc_type"] not in PROMPTS_EXTRACTION:
            record["extraction_status"] = "NON_APPLICABLE"
            continue

        try:
            extract_one_record(record, pdf_path)
        except Exception as exc:
            record["extraction_status"] = "ERREUR"
            record["extraction_error"] = repr(exc)

    return records

def process_pdf(pdf_path, verbose=True):
    t0 = time.time()

    if verbose:
        log(f"🔍 {pdf_path.name}")

    pages = pdf_to_pages(pdf_path)
    if not pages:
        raise ValueError(f"Aucune page détectée dans {pdf_path.name}")

    records = classify_pages(pages)
    records = extract_classified_pages(records, pdf_path)

    page_rows = [
        build_page_row(pdf_path.name, record)
        for record in records
    ]

    dossier_row = consolidate_dossier(pdf_path.name, records)
    planning = []

    tokens_in = sum(
        record.get("classification_tokens_in", 0)
        + record.get("extraction_tokens_in", 0)
        for record in records
    )
    tokens_out = sum(
        record.get("classification_tokens_out", 0)
        + record.get("extraction_tokens_out", 0)
        for record in records
    )

    elapsed_total = round(time.time() - t0, 3)

    dossier_row.update({
        "STATUT_TRAITEMENT_PIPELINE": "TRAITE_NOUVEAU",
        "TEMPS_ECOULE_DOSSIER_S": round(elapsed_total, 2),
        "TOKENS_IN_DOSSIER": int(tokens_in),
        "TOKENS_OUT_DOSSIER": int(tokens_out),
        "TOKENS_TOTAL_DOSSIER": int(tokens_in + tokens_out),
    })

    dossier = {
        "source_file": pdf_path.name,
        "source_sha256": sha256_file(pdf_path),
        "pipeline_version": PIPELINE_VERSION,
        "stats": {
            "pages": len(pages),
            "tokens_in": tokens_in,
            "tokens_out": tokens_out,
            "tokens_total": tokens_in + tokens_out,
            "elapsed_s": elapsed_total,
        },
        "page_records": [
            {
                key: value
                for key, value in record.items()
                if key != "image"
            }
            for record in records
        ],
        "page_rows": page_rows,
        "dossier_row": dossier_row,
        "planning_tl": planning,
    }

    checkpoint = canonical_checkpoint_path(pdf_path)
    with open(checkpoint, "w", encoding="utf-8") as f:
        json.dump(dossier, f, ensure_ascii=False, indent=2, default=str)

    return dossier

print("✅ Classification V7.1 (requalification planche permis) OK")
print("✅ Extraction V7.1 avec escalade et fusion multi-recadrages OK")

In [ ]:
def ordered_columns(rows):
    preferred = [
        "FICHIER", "NB_PAGES", "TYPES_DOCUMENTS",
        "STATUT_TRAITEMENT_PIPELINE", "TEMPS_ECOULE_DOSSIER_S",
        "TOKENS_IN_DOSSIER", "TOKENS_OUT_DOSSIER", "TOKENS_TOTAL_DOSSIER",
        "REFERENCE_DOM_EXTRAITE_RAW", "REFERENCE_DOM_EXTRAITE_NORMALISEE",
        "NUMERO_DOM_RETENU", "DATE_DOM_RETENUE",
        "REFERENCE_PREDOM_RETENUE", "DATE_DEBUT_CONTRAT_PREDOM",
        "DATE_FIN_CONTRAT_PREDOM", "PREDOM_TROUVEE", "NUMERO_CLIENT_RETENU",
        "NOM_CLIENT_REFERENCE", "NUMERO_CONTRAT_REFERENCE",
        "DATE_DEBUT_CONTRAT_REFERENCE", "DATE_FIN_CONTRAT_REFERENCE",
        "NUMERO_PERMIS_REFERENCE", "MATCH_SOURCE", "MATCH_METHOD",
        "MATCH_SCORE", "MATCH_STATUS", "MATCH_CANDIDATES_COUNT",
        "REFERENCE_PREDOM_RETENUE", "DATE_DEBUT_CONTRAT_PREDOM",
        "DATE_FIN_CONTRAT_PREDOM", "PREDOM_TROUVEE",
        "-- V7.1 : augmentation de salaire --",
        "SALAIRE_ANCIEN", "SALAIRE_NOUVEAU_AUGMENTE",
        "AUGMENTATION_DETECTEE", "SENS_VARIATION_SALAIRE",
        "MONTANT_AUGMENTATION", "TAUX_AUGMENTATION_PCT",
        "SOURCE_AUGMENTATION", "CTS_LIGNE_SALAIRE_BRUTE",
        "ECART_SALAIRE_DOM_CONTRAT", "COHERENCE_SALAIRE_DOM_CONTRAT",
        "ALERTE_DOM_SUR_ANCIEN_SALAIRE",
        "-- V7.1 : identité et permis de travail --",
        "NOM_TRAVAILLEUR_REFERENCE", "DATE_NAISSANCE_REFERENCE",
        "NATIONALITE_REFERENCE", "DATE_ENTREE_ALGERIE",
        "PERMIS_TRAVAIL_LU",
    ]
    cols = set().union(*(r.keys() for r in rows)) if rows else set()
    return [c for c in preferred if c in cols] + sorted(cols - set(preferred))

def sheet_from_rows(wb, title, rows, columns=None, amount_columns=None):
    ws = wb.create_sheet(title)
    columns = columns or ordered_columns(rows)
    if not columns:
        ws["A1"] = "Aucune donnée"
        return

    amount_columns = set(amount_columns or [])
    fill = PatternFill("solid", fgColor="1F4E78")
    font = Font(color="FFFFFF", bold=True, name="Arial", size=9)

    for j, name in enumerate(columns, 1):
        c = ws.cell(1, j, name)
        c.fill = fill
        c.font = font
        c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

    for i, row in enumerate(rows, 2):
        for j, name in enumerate(columns, 1):
            value = row.get(name)
            if isinstance(value, (dict, list)):
                value = json.dumps(value, ensure_ascii=False)
            c = ws.cell(i, j, value)
            if name in amount_columns and isinstance(value, (int, float)):
                c.number_format = EXCEL_AMOUNT_FORMAT

    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions
    for j, name in enumerate(columns, 1):
        ws.column_dimensions[get_column_letter(j)].width = 38 if "ADRESSE" in name else min(34, max(14, len(name) + 2))

def build_long_raw_rows(dossiers):
    rows = []
    for d in dossiers:
        for page in d.get("page_records", []):
            for field, value in (page.get("raw_data") or {}).items():
                rows.append({
                    "FICHIER": d.get("source_file"),
                    "PAGE": page.get("page_num"),
                    "TYPE_DOCUMENT": page.get("doc_type"),
                    "CHAMP": field,
                    "VALEUR_BRUTE": value,
                    "CONFIANCE_CLASSIFICATION": page.get("classification_confidence"),
                    "STATUT_EXTRACTION": page.get("extraction_status"),
                })
    return rows

COLONNES_MONTANT_V71 = [
    "SALAIRE_ANCIEN", "SALAIRE_NOUVEAU_AUGMENTE",
    "MONTANT_AUGMENTATION", "ECART_SALAIRE_DOM_CONTRAT",
]

def create_excel(excel_path, dossiers, errors, reference_df=None):
    pages, dossier_rows, planning, matching = [], [], [], []

    for d in dossiers:
        pages.extend(d.get("page_rows", []))
        row = d.get("dossier_row") or {}
        dossier_rows.append(row)
        planning.extend(d.get("planning_tl", []))
        matching.append({k: row.get(k) for k in [
            "FICHIER", "STATUT_TRAITEMENT_PIPELINE",
            "TEMPS_ECOULE_DOSSIER_S", "TOKENS_IN_DOSSIER",
            "TOKENS_OUT_DOSSIER", "TOKENS_TOTAL_DOSSIER",
            "NOM_CLIENT_REFERENCE", "NUMERO_CONTRAT_REFERENCE",
            "DATE_DEBUT_CONTRAT_REFERENCE", "DATE_FIN_CONTRAT_REFERENCE",
            "REFERENCE_DOM_EXTRAITE_RAW", "REFERENCE_DOM_EXTRAITE_NORMALISEE",
            "MATCH_SOURCE", "MATCH_METHOD", "MATCH_SCORE", "MATCH_STATUS",
            "MATCH_CANDIDATES_COUNT", "NUMERO_CLIENT_RETENU",
            "NUMERO_DOM_RETENU", "DATE_DOM_RETENUE",
            "REFERENCE_PREDOM_RETENUE", "DATE_DEBUT_CONTRAT_PREDOM",
            "DATE_FIN_CONTRAT_PREDOM", "PREDOM_TROUVEE",
            "AUGMENTATION_DETECTEE", "ALERTE_DOM_SUR_ANCIEN_SALAIRE",
        ]})

    raw = build_long_raw_rows(dossiers)
    suivi = []
    for d in dossiers:
        stats = d.get("stats") or {}
        row = d.get("dossier_row") or {}
        suivi.append({
            "FICHIER": d.get("source_file"),
            "STATUT_TRAITEMENT_PIPELINE": row.get("STATUT_TRAITEMENT_PIPELINE"),
            "NB_PAGES": stats.get("pages"),
            "TEMPS_ECOULE_DOSSIER_S": row.get("TEMPS_ECOULE_DOSSIER_S"),
            "TOKENS_IN_DOSSIER": row.get("TOKENS_IN_DOSSIER"),
            "TOKENS_OUT_DOSSIER": row.get("TOKENS_OUT_DOSSIER"),
            "TOKENS_TOTAL_DOSSIER": row.get("TOKENS_TOTAL_DOSSIER"),
            "PIPELINE_VERSION_JSON": d.get("pipeline_version"),
            "SHA256": d.get("source_sha256"),
        })

    wb = Workbook()
    wb.remove(wb.active)

    sheet_from_rows(wb, "DOSSIERS_DOMICILIATION", dossier_rows,
                    amount_columns=AMOUNT_FIELDS | COLONNES_MONTANT_V71)
    sheet_from_rows(wb, "SUIVI_TRAITEMENT", suivi, [
        "FICHIER", "STATUT_TRAITEMENT_PIPELINE", "NB_PAGES",
        "TEMPS_ECOULE_DOSSIER_S", "TOKENS_IN_DOSSIER",
        "TOKENS_OUT_DOSSIER", "TOKENS_TOTAL_DOSSIER",
        "PIPELINE_VERSION_JSON", "SHA256",
    ], amount_columns={"TEMPS_ECOULE_DOSSIER_S"})

    sheet_from_rows(wb, "PLANNING_TL", planning, amount_columns={
        "SALAIRE_NET_REFERENCE", "PLAFOND_MENSUEL_REFERENCE",
        "MONTANT_MAX_THEORIQUE", "MONTANT_AUTORISE_SAISI",
        "MONTANT_TRANSFERE", "SOLDE_RESTANT", "NB_JOURS_SEGMENT", "NB_JOURS_MOIS", "COEFFICIENT_PRORATA",
        "SALAIRE_ANCIEN", "SALAIRE_NOUVEAU_AUGMENTE", "MONTANT_AUGMENTATION",
    })

    sheet_from_rows(wb, "PAGES_DOCUMENTS", pages)
    sheet_from_rows(wb, "EXTRACTION_BRUTE", raw, [
        "FICHIER", "PAGE", "TYPE_DOCUMENT", "CHAMP", "VALEUR_BRUTE",
        "CONFIANCE_CLASSIFICATION", "STATUT_EXTRACTION",
    ])
    sheet_from_rows(wb, "MATCHING_DOM", matching)
    sheet_from_rows(wb, "ERREURS", errors)

    if reference_df is not None:
        sheet_from_rows(wb, "REFERENTIEL_DOM_PREDOM", reference_df.where(pd.notna(reference_df), None).to_dict("records"))

    wb.save(excel_path)
    print(f"✅ Excel créé : {excel_path} | dossiers={len(dossier_rows)} | planning={len(planning)}")

print("✅ Export V6.5 chargé avec onglet SUIVI_TRAITEMENT")

In [ ]:
print(f"Nombre de PDF détectés : {len(pdfs)}")
if not pdfs:
    raise RuntimeError(
        f"Aucun PDF trouvé dans {INPUT_DIR}. "
        "Déposer les dossiers de domiciliation dans ce répertoire."
    )

diagnostic_errors = []
total_pages = 0

for p in pdfs:
    try:
        with fitz.open(str(p)) as doc:
            page_count = int(doc.page_count)
            total_pages += page_count
            print(
                f"{p.name} -> pages={page_count}, "
                f"taille={p.stat().st_size:,} octets"
            )
            if page_count <= 0:
                diagnostic_errors.append(f"{p.name}: 0 page")
    except Exception as exc:
        diagnostic_errors.append(f"{p.name}: {exc!r}")

if diagnostic_errors:
    raise RuntimeError(
        "Diagnostic PDF en erreur :\n" + "\n".join(diagnostic_errors)
    )

print(f"✅ Diagnostic PDF : {len(pdfs)} fichier(s), {total_pages} page(s)")

# Nettoyage des checkpoints invalides.
removed = 0
for json_file in JSON_DIR.glob("*.json"):
    try:
        dossier = json.loads(json_file.read_text(encoding="utf-8"))
        pages_checkpoint = int(dossier.get("stats", {}).get("pages", 0) or 0)
        records_checkpoint = dossier.get("page_records") or []
        if pages_checkpoint <= 0 or not records_checkpoint:
            json_file.unlink()
            removed += 1
            print(f"Checkpoint vide supprimé : {json_file.name}")
    except Exception:
        json_file.unlink()
        removed += 1
        print(f"Checkpoint illisible supprimé : {json_file.name}")

print(f"Checkpoints invalides supprimés : {removed}")

In [ ]:
ram_free = psutil.virtual_memory().available / 1_000_000_000
log(f"RAM libre : {ram_free:.1f} GB")
reference_df = build_reference_table()
log(f"Référentiel externe : {0 if reference_df is None else len(reference_df)} ligne(s)")

all_dossiers, errors = [], []
nb_repris = 0
nb_nouveaux = 0
pipeline_start = time.time()

print_pipeline_header(len(pdfs))

for position, pdf_path in enumerate(pdfs, 1):
    try:
        dossier = load_existing_checkpoint(pdf_path)
        skipped = dossier is not None

        if skipped:
            dossier = enrich_dossier_row_with_stats(
                dossier,
                "REPRIS_JSON_EXISTANT",
            )
            nb_repris += 1
        else:
            dossier = process_pdf(pdf_path, verbose=False)
            dossier = enrich_dossier_row_with_stats(
                dossier,
                "TRAITE_NOUVEAU",
            )
            nb_nouveaux += 1

        all_dossiers.append(dossier)

        stats = dossier.get("stats") or {}
        print_compact_progress(
            position=position,
            total=len(pdfs),
            pdf_name=pdf_path.name,
            pages=stats.get("pages", 0),
            skipped=skipped,
            tokens_in=stats.get("tokens_in", 0),
            tokens_out=stats.get("tokens_out", 0),
            elapsed_s=stats.get("elapsed_s", 0),
            pipeline_start=pipeline_start,
        )

    except Exception as exc:
        errors.append({
            "FICHIER": pdf_path.name,
            "ETAPE": "PROCESS_PDF",
            "ERREUR": repr(exc),
            "DATE": datetime.now().isoformat(timespec="seconds"),
        })
        print(
            f"[{position}/{len(pdfs)}] "
            f"{pdf_path.name} | ERREUR | {exc!r}",
            flush=True,
        )
        log(f"[{position}/{len(pdfs)}] ❌ {pdf_path.name} : {exc!r}")

    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

for dossier in all_dossiers:
    row = dossier.get("dossier_row") or {}
    row.update(match_dossier(row, reference_df))
    dossier["dossier_row"] = row

    source_pdf = INPUT_DIR / dossier.get("source_file", "")
    if source_pdf.exists():
        canonical_checkpoint_path(source_pdf).write_text(
            json.dumps(dossier, ensure_ascii=False, indent=2, default=str),
            encoding="utf-8",
        )

create_excel(EXCEL_PATH, all_dossiers, errors, reference_df)

with open(MASTER_JSON_PATH, "w", encoding="utf-8") as f:
    json.dump({
        "generated_at": datetime.now().isoformat(timespec="seconds"),
        "pipeline_version": PIPELINE_VERSION,
        "dossiers": all_dossiers, "errors": errors,
    }, f, ensure_ascii=False, indent=2, default=str)

elapsed_pipeline = time.time() - pipeline_start
total_tokens_in = sum(
    int((d.get("stats") or {}).get("tokens_in", 0) or 0)
    for d in all_dossiers
)
total_tokens_out = sum(
    int((d.get("stats") or {}).get("tokens_out", 0) or 0)
    for d in all_dossiers
)

print(
    f"\nTerminé | total={len(all_dossiers)} "
    f"| traités={nb_nouveaux} "
    f"| skip={nb_repris} "
    f"| erreurs={len(errors)} "
    f"| IN={total_tokens_in:,} "
    f"| OUT={total_tokens_out:,} "
    f"| durée={format_duration(elapsed_pipeline)}",
    flush=True,
)

log(f"✅ Pipeline terminé | dossiers={len(all_dossiers)} | nouveaux={nb_nouveaux} | repris={nb_repris} | erreurs={len(errors)}")